# 베이스라인

## 🚀 RAG(Retrieval-Augmented Generation) 시스템 구축 가이드라인

효율적인 정보 검색과 정확한 답변 생성을 위한 RAG 파이프라인 구축 단계별 핵심 요약입니다.

<br>

### ✂️ 1. 데이터 전처리: 로드 및 청킹 (Chunking)

*   **문서 분할:** 검색 효율을 위해 문서를 적절한 길이(Chunk)로 나눕니다.

*   **최적화 실험:** 단순 글자 수 기반 분할 외에 **문서 구조(헤더, 문단)** 나 **의미 단위** 를 고려한 청킹 전략을 시도하며 최적의 옵션을 찾습니다.

### 🧠 2. 임베딩 및 벡터 DB 저장

*   **모델 선정:** Hugging Face의 다양한 임베딩 모델을 비교합니다. 특히 **한국어 성능**이 검증된 모델(예: BGE-M3, KoSimCSE 등)을 우선 고려하세요.

*   **저장소 구축:** 구현 편의성과 검색 속도를 고려하여 적합한 **벡터 데이터베이스**(Chroma, FAISS, Pinecone 등)를 선택해 임베딩을 저장합니다.

### 🤖 3. LLM 및 토크나이저 설정

*   **모델 선택:** 한국어 추론 능력이 뛰어난 모델을 선정합니다.

*   **경량화:** 메모리 부하를 줄이고 속도를 높이기 위해 **양자화(Quantization)** 기법 적용을 검토합니다.

*   **파라미터 튜닝:** `Temperature`, `Penalty` 등 생성 옵션을 조정하여 답변의 창의성과 일관성을 제어합니다.

### ✅ 4. RAG 시스템 구현 및 평가

*   **맥락(Context) 활용:** 질문과 연관된 청크를 찾아 LLM에 전달하는 전체 파이프라인을 구축합니다.

*   **성능 검증:** 문서 내용에 기반한 구체적인 질문(예: "2024 개정 세법 중 월세 관련 내용")을 통해 답변의 **정확성**과 **신뢰도**를 평가합니다.

### 🔥 5. (심화) 고도화 및 확장 실험

*   **Advanced RAG:** 검색 품질 개선을 위해 아래 기법들을 적용해 봅니다.

    1.   **기본 RAG**
    2.   **Hybrid Search:** 키워드(BM25) + 문맥(벡터 검색) 결합 $\to$ 공부해보기!
    3.   **Multi-Query Retrieval:** 질문 재구성 및 확장
    4.   **Reranking:** 검색 결과의 재순위화
    5.   **Contextual Compression**

*   **LLM 비교:** Hugging Face 모델 외에도 **OpenAI API** 등 상용 모델과의 성능 차이를 비교 분석합니다.

## 구글 드라이브 마운트

In [5]:
from google.colab import drive
drive.mount('/content/drive', force_remount=True)

Mounted at /content/drive


## 라이브러리 설치

In [ ]:
!pip uninstall -y faiss-cpu # 기존에 설치된 faiss-cpu 패키지를 제거합니다.

!pip install faiss-gpu-cu12 # CUDA 12에 맞는 faiss-gpu 패키지를 설치합니다. CUDA 버전에 따라 적절한 패키지를 선택해야 합니다.


In [15]:
!nvcc --version

nvcc: NVIDIA (R) Cuda compiler driver
Copyright (c) 2005-2025 NVIDIA Corporation
Built on Fri_Feb_21_20:23:50_PST_2025
Cuda compilation tools, release 12.8, V12.8.93
Build cuda_12.8.r12.8/compiler.35583870_0


In [ ]:
# NVIDIA GPU 사용시, faiss-gpu-cu12 설치

!pip install \
    "langchain>=0.3.0,<0.4.0" \
    "langchain-community>=0.3.0,<0.4.0" \
    "langchain-core>=0.3.0,<0.4.0" \
    langchain-huggingface \
    langchain-openai \
    langchain-text-splitters \
    faiss-gpu-cu12 tiktoken==0.9.0 \
    datasets \
    -q


## 문서 파일 불러오기

In [7]:
!ls /content/drive/MyDrive/data/rag_files/

year_end_tax.pdf


In [9]:
!pip install pypdf

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 336.3/336.3 kB 29.5 MB/s eta 0:00:00


In [10]:
from langchain_community.document_loaders import PyPDFLoader

file_path = "/content/drive/MyDrive/data/rag_files/year_end_tax.pdf" # PDF 파일 경로를 지정해주세요.
loader = PyPDFLoader(file_path) # PDF 파일을 로드하는 로더 객체를 생성합니다.
pages = []
async for page in loader.alazy_load(): # PDF 파일의 각 페이지를 비동기적으로 로드하여 pages 리스트에 추가합니다.
    pages.append(page)

In [11]:
pages[0].page_content[:500]

'연말정산\n신고안내\n일 하나는 제대로 하는,\n국민께 인정받는 국세청\n2024. 12.\n2024 원천징수의무자를 위한\n맞춤형 안내\n간소화 서비스\n 일괄제공 서비스\n발간등록번호\n11-1210000-000072-10'

## 문서 청킹

In [12]:
from langchain_text_splitters import RecursiveCharacterTextSplitter # 텍스트를 일정한 크기로 나누는 도구입니다. 긴 문서를 작은 조각으로 분할하여 처리할 수 있도록 도와줍니다.

text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=500, # 각 조각의 최대 길이를 500자로 설정합니다. 문서가 이 길이를 초과하면 여러 조각으로 나뉩니다.
    chunk_overlap=100, # 조각 간의 겹치는 부분을 100자로 설정합니다. 이렇게 하면 문맥이 끊어지는 것을 방지할 수 있습니다.
    length_function=len, # 텍스트의 길이를 계산하는 함수로, 기본적으로 Python의 내장 len 함수를 사용합니다.
)
all_splits = text_splitter.split_documents(pages) # pages 리스트에 있는 각 페이지를 text_splitter를 사용하여 조각으로 나눕니다. 결과는 all_splits 리스트에 저장됩니다.
print(f"총 {len(all_splits)}개의 조각이 생성되었습니다.") # 생성된 조각의 총 개수를 출력합니다.
print("")
print(all_splits[0].page_content[:500]) # 첫 번째 조각의 내용

총 1119개의 조각이 생성되었습니다.

연말정산
신고안내
일 하나는 제대로 하는,
국민께 인정받는 국세청
2024. 12.
2024 원천징수의무자를 위한
맞춤형 안내
간소화 서비스
 일괄제공 서비스
발간등록번호
11-1210000-000072-10


In [13]:
print(all_splits[6].page_content[:500]) # 첫 번째 조각의 내용

「연말정산 신고안내」 책자 및 동영상
  「편리한 연말정산 이용방법」 동영상
국세청 홈페이지(www.nts.go.kr) 
국세신고안내  개인 또는 법인  연말정산
프로그램   퇴직소득 세액계산 프로그램
  지급명세서 전산매체 제출요령
국세청 홈택스(www.hometax.go.kr) 
자료실


## Vector DB에 저장

* KURE: Korea University Retrieval Embedding model : https://github.com/nlpai-lab/KURE

#### 🛠️ GPU 가속을 위한 핵심 변경점 3단계

##### 1. GPU 자원 할당 (`StandardGpuResources`)
```python
res = faiss.StandardGpuResources()
```
*   **역할:** FAISS가 GPU의 비디오 메모리(VRAM)를 효율적으로 사용할 수 있도록 관리 객체를 생성합니다. 이 객체가 없으면 GPU 인덱스를 만들 수 없습니다.

##### 2. CPU 인덱스를 GPU로 복사 (질문하신 핵심 로직)
```python
gpu_index = faiss.index_cpu_to_gpu(res, 0, cpu_index)
```
*   **역할:** 메모리에 생성된 `IndexFlatL2` 구조를 **GPU 0번(L4)** 으로 보냅니다.
*   **효과:** 이제부터 발생하는 모든 벡터 비교(L2 거리 계산) 연산은 CPU가 아닌 **GPU의 병렬 연산 코어**를 사용하게 됩니다.

##### 3. 벡터 스토어 주입
```python
vector_store = FAISS(
    ...,
    index=gpu_index, # 변환된 인덱스 사용
    ...
)
```
*   **역할:** LangChain의 `FAISS` 클래스가 CPU 인덱스 대신 GPU 인덱스를 바라보도록 연결합니다.

<br>

##### ⚠️ 주의사항 및 체크포인트

단순히 코드만 추가한다고 바로 작동하지 않을 수 있으니 다음 사항을 꼭 확인해 보세요.

1.  **라이브러리 설치:** 일반 `faiss-cpu`가 아닌 **`faiss-gpu-cu12`** 패키지가 설치되어 있어야 합니다.
    
    *   `pip install faiss-gpu-cu12`

2.  **임베딩 생성 시점:** 

    *   `vector_store.add_documents()`를 실행하면 `HuggingFaceEmbeddings` 모델이 텍스트를 벡터로 만듭니다. 

    *   이때 **임베딩 모델 자체**도 GPU를 쓰게 하려면 `model_kwargs={'device':'cuda'}` 옵션을 `HuggingFaceEmbeddings` 생성 시 추가하는 것이 훨씬 빠릅니다.

3.  **병목 현상:**

    *   위 코드는 **검색(Search)** 단계를 GPU로 가속한 것입니다. 만약 문서 개수가 수만 개 수준이라면 체감이 크지만, 수백 개 수준이라면 데이터를 GPU로 옮기는 오버헤드 때문에 오히려 CPU보다 느릴 수도 있습니다. (우리는 1192개로 체감 속도는 나타나지 않을 수 있음.)


In [ ]:
import faiss
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_community.docstore.in_memory import InMemoryDocstore
from langchain_community.vectorstores import FAISS

# 1. 임베딩 모델 설정
# embedding_model_name = "nlpai-lab/KURE-v1" # 더 긴 문장 단위로 청킹을 해보고 싶다면, 이 모델을 사용해보세요!
embedding_model_name = "nlpai-lab/KoE5"
embeddings = HuggingFaceEmbeddings(model_name=embedding_model_name)

# 2. 임베딩 차원 계산 및 초기 CPU 인덱스 생성
embedding_dim = len(embeddings.embed_query("hello world")) 
cpu_index = faiss.IndexFlatL2(embedding_dim)

# 3. GPU 리소스 초기화 및 인덱스 이동 (핵심)
# StandardGpuResources는 GPU 메모리 자원을 관리합니다.
res = faiss.StandardGpuResources()
# cpu_index를 GPU 0번(L4)으로 복사합니다.
gpu_index = faiss.index_cpu_to_gpu(res, 0, cpu_index)

# 4. FAISS 벡터 스토어 생성 (GPU 인덱스 주입)
vector_store = FAISS( 
    embedding_function=embeddings,
    index=gpu_index,              # 변환된 gpu_index를 사용합니다.
    docstore=InMemoryDocstore(),
    index_to_docstore_id={},
)

# 5. 문서 추가 (L4 GPU 가속을 통해 벡터가 인덱싱됩니다)
ids = vector_store.add_documents(documents=all_splits)

print(f"총 {len(ids)}개의 조각이 GPU 벡터 스토어에 추가되었습니다.")

# 6. 검색기(Retriever) 생성
retriever = vector_store.as_retriever()


In [15]:
ids

['b4672995-705b-40db-910d-76e42fdd4ddb',
 '454e54d8-6690-40f1-bceb-dd46cb820282',
 '43ac6600-7c09-4410-a890-cd01e9f2986b',
 '633d4303-0ebb-4d67-b0b1-0be40d644444',
 '3ac42fb1-a72b-47bd-a76e-167c639c50aa',
 'c153bd66-f76f-4e18-ac96-ac0c64a770f9',
 '2130c0a5-68d6-49c0-8497-5dae24c75dbb',
 '0c656f51-59ce-44df-834e-068d1af8cff1',
 '8c7aad74-1464-4467-9fe6-309b345f3cc4',
 '79706661-ecc8-470b-bc54-4b816a9ca819',
 '5f107b7c-e0d3-460a-87c8-a099089e8892',
 '6bfe306d-0644-4853-9d02-9ba65c6ea89c',
 '08d5355e-5457-44ae-add3-5a7547f8fe2d',
 '5a963305-36d7-4a85-a033-0ffc24b6a43d',
 'fa3205c5-1e8a-4d21-a5bb-5a9485f44242',
 'c644d832-4322-457d-b68a-39dfa7202eeb',
 'b6dbf877-8370-4297-8f66-cf9a4276938f',
 'c130f92f-b961-4fa8-a248-51c93d6d1b1f',
 '41c3d214-870a-4b40-b4c1-527ce5ec3f1c',
 '5d04d612-d081-4986-8291-5f8e790ee593',
 'f621b16c-7f8a-4978-822c-8d247332b343',
 'f1d6e633-2f88-4219-b21a-1f1d3197403c',
 '41cc9233-7eaa-4d15-8727-ae559dd15770',
 'ec6d2ea3-3c4d-4a04-bf83-a2d57707fb63',
 '0ce90aff-43a3-

## 모델 및 토크나이저 세팅

In [ ]:
from langchain_huggingface import ChatHuggingFace, HuggingFacePipeline
from langchain.schema.runnable import RunnablePassthrough
from langchain.schema.output_parser import StrOutputParser
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig, pipeline
import torch


#language_model_name = "nlpai-lab/KULLM3" # 한국어에 특화된 대형 언어 모델로, 자연어 처리 작업에서 높은 성능을 발휘합니다. 이 모델은 한국어 텍스트의 의미를 잘 이해하고 생성할 수 있도록 훈련되어 있습니다. RAG 시스템에서 질문에 대한 답변을 생성하는 데 사용됩니다.
# language_model_name = "Bllossom/llama-3.2-Korean-Bllossom-AICA-5B"


bnb_config = BitsAndBytesConfig( # 모델을 4비트 양자화하여 메모리 사용량을 줄이는 설정입니다. 이 설정은 모델의 크기를 줄이고, 더 작은 GPU에서도 실행할 수 있도록 도와줍니다. 양자화는 모델의 정확도에 영향을 줄 수 있지만, 적절한 설정을 통해 성능 저하를 최소화할 수 있습니다.
    load_in_4bit=True,
    bnb_4bit_use_double_quant=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.float16,
    llm_int8_enable_fp32_cpu_offload=True,
)
model = AutoModelForCausalLM.from_pretrained( # 사전 훈련된 언어 모델을 로드합니다. Hugging Face의 Transformers 라이브러리를 사용하여 모델을 불러오며, 양자화 설정을 적용하여 메모리 효율성을 높입니다. 
    language_model_name,
    quantization_config=bnb_config,
    trust_remote_code=True, # 원격 저장소에(허깅페이스 허브)서 사용자 정의 모델 코드를 신뢰하도록 설정합니다. 이 옵션은 모델이 Hugging Face Hub에 업로드된 경우에 필요할 수 있습니다. 모델의 코드가 안전하다고 확신하는 경우에만 이 옵션을 활성화해야 합니다.
)
tokenizer = AutoTokenizer.from_pretrained(language_model_name)


In [ ]:
from transformers import pipeline

llm_pipeline = pipeline( # Hugging Face의 Transformers 라이브러리를 사용하여 텍스트 생성 파이프라인을 설정합니다. 이 파이프라인은 사전 훈련된 언어 모델을 사용하여 입력된 텍스트에 대한 응답을 생성하는 데 사용됩니다. 다양한 생성 옵션을 설정하여 출력의 다양성과 품질을 조절할 수 있습니다.
    model=model,
    tokenizer=tokenizer,
    task="text-generation",
    do_sample=True,
    temperature=0.2,        # RAG 사용시에는 일관적인 답변이 나와야 하므로 0.2와 같은 작은 값을 사용합니다.
    repetition_penalty=1.2, # 생성된 텍스트에서 반복되는 단어의 패널티를 설정합니다. 값이 1보다 크면 반복되는 단어가 덜 생성됩니다.
    return_full_text=False, # 생성된 텍스트에서 입력 프롬프트를 포함할지 여부를 설정합니다. False로 설정하면 출력에서 입력 프롬프트가 제외됩니다.
    max_new_tokens=1000,    # 생성할 최대 토큰 수를 설정합니다. 이 값은 모델이 생성할 수 있는 텍스트의 길이를 제한합니다.
)
llm = HuggingFacePipeline(pipeline=llm_pipeline)

# Hugging Face의 텍스트 생성 파이프라인을 래핑하여 대화형 모델로 사용할 수 있도록 합니다. 이 모델은 RAG 시스템에서 질문에 대한 답변을 생성하는 데 사용됩니다. 
# 입력된 쿼리에 대해 관련 문서를 검색한 후, 해당 문서를 기반으로 답변을 생성할 수 있습니다.
chat_model = ChatHuggingFace(llm=llm) 


## RAG

In [ ]:
from langchain_core.prompts import PromptTemplate


template = """
아래에 주어진 맥락을 이용해 질문에 대해 답변해 줘.
주어진 맥락으로 답변이 어려운 상황이라면, 그냥 모른다고 답하면 되고 억지로 답변을 꾸며 내지 마.
최대한 자세하게 답변해 줘.
반드시 한국어로 답변해야 해.

맥락:
{context}

질문:
{question}
"""

prompt = PromptTemplate.from_template(template) # RAG 시스템에서 사용할 프롬프트 템플릿을 정의합니다. 이 템플릿은 모델이 질문에 답변할 때 사용할 맥락과 질문을 포함하도록 설계되었습니다. 모델이 답변을 생성할 때 이 템플릿을 사용하여 입력을 구성하게 됩니다.  

- LCEL(LangChain Expression Language) 기반의 RAG 체인 코드

In [ ]:
from langchain.schema.runnable import RunnablePassthrough 
from langchain.schema.output_parser import StrOutputParser

# 1. 문서 포맷팅 함수
# 검색기(Retriever)가 찾아온 LangChain Document 객체 리스트를 
# 하나의 문자열(String)로 합쳐주는 역할을 합니다.
def format_docs(docs):
    # 디버깅을 위해 검색된 문서들을 출력합니다.
    print(docs) 
    # 각 문서의 page_content(실제 내용) 사이에 줄바꿈 두 번을 넣어 결합합니다.
    return "\n\n".join(doc.page_content for doc in docs)

# 2. RAG 실행 체인 정의
retrieval_chain = (
    # [A단계] 입력 데이터 준비 (Dictionary 구성)
    # 체인이 시작될 때 두 가지 데이터를 준비합니다.
    {
        # 1) 검색기(retriever)를 통해 관련 문서를 찾고, format_docs로 문자열 변환하여 'context'에 할당
        "context": retriever | format_docs, # 각 컴포넌트가 파이프(|) 연산자로 연결되서, 순차적으로 처리된다라는 뜻.
        
        # 2) 사용자의 질문(입력값)을 변형 없이 그대로 'question'에 할당
        "question": RunnablePassthrough()
    }
    
    # [B단계] 프롬프트 생성
    # 앞 단계에서 준비된 {'context': ..., 'question': ...} 딕셔너리가 prompt 템플릿으로 전달됩니다.
    # 템플릿 내의 {context}와 {question} 변수에 값이 자동으로 주입됩니다.
    | prompt
    
    # [C단계] 모델 호출
    # 완성된 프롬프트 문자열이 LLM(언어 모델)으로 전달되어 답변이 생성됩니다.
    | llm
    
    # [D단계] 출력 결과 파싱
    # 모델의 응답 객체(AIMessage)에서 텍스트 내용만 뽑아내어 최종 문자열로 변환합니다.
    | StrOutputParser()
)

In [ ]:
question = "연말 정산 때 비거주자가 주의할 점을 알려 줘." # RAG 시스템에 입력할 질문입니다. 이 질문은 사용자가 RAG 시스템에 묻고자 하는 내용을 나타냅니다. 시스템은 이 질문을 기반으로 관련 문서를 검색하고, 해당 문서를 활용하여 답변을 생성하게 됩니다.

result = retrieval_chain.invoke(question) # RAG 시스템을 실행하여 질문에 대한 답변을 생성합니다. 이 과정에서 검색된 문서가 프롬프트에 삽입되고, 모델이 해당 프롬프트를 기반으로 답변을 생성하게 됩니다. 최종적으로 생성된 답변이 result 변수에 저장됩니다.
print(result)

[Document(id='a9b89cbb-aadb-49f9-ae12-7fcd4fa97fdf', metadata={'producer': 'ezPDF Builder Supreme', 'creator': 'PyPDF', 'creationdate': '2024-12-22T23:44:00+09:00', 'moddate': '2025-01-09T17:28:20+09:00', 'source': '/content/drive/Shareddrives/스프린트(AI) 드라이브/트랙 Master 폴더/스프린트 미션 및 모범답안/data/rag_files/year-end-tax.pdf', 'total_pages': 426, 'page': 102, 'page_label': '103'}, page_content='확정신고기한까지 신고납부한 것으로 본다.(소령 §134④)\n05 비거주자의 연말정산\n가. 거주자와 비거주자\n1) 거주자와 비거주자 (소법 §1의2)\n  거주자는 국내에 주소를 두거나 183일 이상의 거소를 둔 개인을 말하며, 비거주자는 거주자가\n아닌 개인을 말한다.\n2) 주소와 거소의 판정 (소령 §2)\n○ 주소는 국내에서 생계를 같이 하는 가족 및 국내에 소재하는 자산의 유무 등 생활관계의 \n객관적 사실에 따라 판정하며, 거소는 주소지 외의 장소 중 상당기간에 걸쳐 거주하는 \n장소로서 주소와 같이 밀접한 일반적 생활관계가 형성되지 아니한 장소이다.\n○ 국내에 거주하는 개인이 국내에 주소를 가진 것으로 보는 경우\n   - 계속하여 183일 이상 국내에 거주할 것을 통상 필요로 하는 직업을 가진 때\n   - 국내에 생계를 같이하는 가족이 있고, 그 직업 및 자산 상태에 비추어 계속하여 183일 이상 \n국내에 거주할 것으로 인정되는 때'), Document(id='37eac7a8-7d70-4dcb-b136-2f0e07db9539', metadata={'producer': 'ezPDF Builder Su

In [ ]:
question = "2024년 개정 세법 중에 월세와 관련한 내용이 있을까?" # RAG 시스템에 입력할 또 다른 질문입니다. 이 질문은 2024년 개정된 세법에서 월세와 관련된 내용이 있는지에 대한 정보를 묻고 있습니다. 시스템은 이 질문을 기반으로 관련 문서를 검색하고, 해당 문서를 활용하여 답변을 생성하게 됩니다.

result = retrieval_chain.invoke(question) # RAG 시스템을 실행하여 질문에 대한 답변을 생성합니다. 이 과정에서 검색된 문서가 프롬프트에 삽입되고, 모델이 해당 프롬프트를 기반으로 답변을 생성하게 됩니다. 최종적으로 생성된 답변이 result 변수에 저장됩니다.
print(result)

[Document(id='7cacc321-4c6d-42c1-8e32-d370bd1d636c', metadata={'producer': 'ezPDF Builder Supreme', 'creator': 'PyPDF', 'creationdate': '2024-12-22T23:44:00+09:00', 'moddate': '2025-01-09T17:28:20+09:00', 'source': '/content/drive/Shareddrives/스프린트(AI) 드라이브/트랙 Master 폴더/스프린트 미션 및 모범답안/data/rag_files/year-end-tax.pdf', 'total_pages': 426, 'page': 32, 'page_label': '33'}, page_content='01. 2024년 귀속 연말정산 개정세법 요약\n17\n24 월세액 세액공제 소득기준 및 한도 상향 \n(조세특례제한법 제95조의2, 제122조의3)\n<개정취지> 서민·중산층 주거비 부담 완화\n종          전 개          정\n▢ 월세 세액공제 ▢ 소득기준 및 한도 상향\n○ (대상) 총급여 7천만원(종합소득금액 6천만원) 이하 무\n주택근로자 및 성실사업자 등\n○ 총급여 8천만원(종합소득금액 7천만원) 이하 무주택\n근로자 및 성실사업자 등\n○ (공제율) 월세액의 15% 또는 17%*\n   ＊ 총급여 5,500만원 또는 종합소득금액 4,500만원 이하자\n○ (좌  동)\n○ (공제한도) 연간 월세액 750만원 ○ 750만원 → 1,000만원\n○ (대상 주택) 국민주택규모(85㎡) 이하 또는 기준시가 \n4억원 이하\n○ (좌  동)\n<적용시기> 2024.1.1. 이후 개시하는 과세연도 분부터 적용\n25 신용카드 소득공제율 한시 상향 등'), Document(id='e8ef67ba-c72b-4ee2-bea4-9bc25177951f', metadata={'producer': 'ezPDF Build

In [ ]:
question = "기부금 공제 때 주의할 점은?" # RAG 시스템에 입력할 또 다른 질문입니다. 이 질문은 기부금 공제와 관련된 주의사항에 대한 정보를 묻고 있습니다. 시스템은 이 질문을 기반으로 관련 문서를 검색하고, 해당 문서를 활용하여 답변을 생성하게 됩니다.

result = retrieval_chain.invoke(question) # RAG 시스템을 실행하여 질문에 대한 답변을 생성합니다. 이 과정에서 검색된 문서가 프롬프트에 삽입되고, 모델이 해당 프롬프트를 기반으로 답변을 생성하게 됩니다. 최종적으로 생성된 답변이 result 변수에 저장됩니다.
print(result)

[Document(id='6fe931b5-3714-4d3a-a412-3e463908b11d', metadata={'producer': 'ezPDF Builder Supreme', 'creator': 'PyPDF', 'creationdate': '2024-12-22T23:44:00+09:00', 'moddate': '2025-01-09T17:28:20+09:00', 'source': '/content/drive/Shareddrives/스프린트(AI) 드라이브/트랙 Master 폴더/스프린트 미션 및 모범답안/data/rag_files/year-end-tax.pdf', 'total_pages': 426, 'page': 244, 'page_label': '245'}, page_content='본인 800,000 500,000 200,000 100,000\n배우자\n직계비속\n직계존속\n형제자매\n그외\n❹ 기부금 조정 명세\n기부금\n코드\n기부\n연도 \x00\x00 기부금액 ⑰ 전년까지\n공제된 금액\n⑱ 공제대상\n금액(\x00\x00-⑰)\n해당 연도 공제금액 해당 연도에 공제받지 못한 금액\n필요경비 세액(소득)공제 소멸금액 이월금액\n10 2023 500,000 - 500,000 - 500,000\n20 2023 200,000 - 200,000 - 200,000\n43 2023 100,000 100,000 100,000'), Document(id='3edca478-2524-4e6e-a9bd-39242b7b6902', metadata={'producer': 'ezPDF Builder Supreme', 'creator': 'PyPDF', 'creationdate': '2024-12-22T23:44:00+09:00', 'moddate': '2025-01-09T17:28:20+09:00', 'source': '/content/drive/Shareddrives/스프린트(AI) 드라이브/트ᄅ

# 미션 14


## 1. 환경 설치 및 드라이브 마운트


In [1]:
from google.colab import drive

drive.mount('/content/drive', force_remount=True)


Mounted at /content/drive


In [2]:
# 미션 14에서는 FAISS 검색은 CPU에 두고, GPU 메모리는 임베딩/LLM에 우선 사용합니다.
# langgraph 계열은 이번 노트북에서 사용하지 않고 LangChain 0.3.x와 충돌 경고를 만들 수 있어 제거합니다.
!pip uninstall -y -q faiss-gpu-cu12 faiss-gpu langgraph langgraph-prebuilt

!pip install -q \
    "requests==2.32.4" \
    "langchain>=0.3.0,<0.4.0" \
    "langchain-community>=0.3.0,<0.4.0" \
    "langchain-core>=0.3.0,<0.4.0" \
    langchain-huggingface \
    langchain-openai \
    langchain-text-splitters \
    faiss-cpu \
    pypdf \
    tiktoken==0.9.0 \
    datasets \
    "bitsandbytes>=0.46.1" \
    accelerate


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.2/1.2 MB 67.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 66.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.5/2.5 MB 98.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 461.3/461.3 kB 46.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 76.0/76.0 kB 9.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.8/23.8 MB 86.7 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 336.3/336.3 kB 32.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.7/60.7 MB 43.8 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 66.5/66.5 kB 8.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 51.0/51.0 kB 6.7 MB/s eta 0:00:00


In [3]:
!nvcc --version


nvcc: NVIDIA (R) Cuda compiler driver
Copyright (c) 2005-2025 NVIDIA Corporation
Built on Fri_Feb_21_20:23:50_PST_2025
Cuda compilation tools, release 12.8, V12.8.93
Build cuda_12.8.r12.8/compiler.35583870_0


## 2. 문서 파일 불러오기


In [4]:
DATA_DIR = "/content/drive/MyDrive/data/rag_files"
!ls {DATA_DIR}


year_end_tax.pdf


In [5]:
from langchain_community.document_loaders import PyPDFLoader

file_path = f"{DATA_DIR}/year_end_tax.pdf"
loader = PyPDFLoader(file_path)

pages = []
async for page in loader.alazy_load():
    pages.append(page)

print(f"로드된 페이지 수: {len(pages)}")
print(pages[0].metadata)
print(pages[0].page_content[:500])


로드된 페이지 수: 426
{'producer': 'ezPDF Builder Supreme', 'creator': 'PyPDF', 'creationdate': '2024-12-22T23:44:00+09:00', 'moddate': '2025-01-09T17:28:20+09:00', 'source': '/content/drive/MyDrive/data/rag_files/year_end_tax.pdf', 'total_pages': 426, 'page': 0, 'page_label': '1'}
연말정산
신고안내
일 하나는 제대로 하는,
국민께 인정받는 국세청
2024. 12.
2024 원천징수의무자를 위한
맞춤형 안내
간소화 서비스
 일괄제공 서비스
발간등록번호
11-1210000-000072-10


## 3. 텍스트 정리 및 메타데이터 유지


In [6]:
from langchain_core.documents import Document


def clean_text(text):
    text = text.replace("\n", " ")
    text = " ".join(text.split())
    return text


cleaned_pages = []

for page in pages:
    metadata = page.metadata.copy()

    if "page" in metadata:
        metadata["page_number"] = metadata["page"] + 1

    cleaned_pages.append(
        Document(
            page_content=clean_text(page.page_content),
            metadata=metadata,
        )
    )

print(f"정리된 페이지 수: {len(cleaned_pages)}")
print(cleaned_pages[0].metadata)
print(cleaned_pages[0].page_content[:500])


정리된 페이지 수: 426
{'producer': 'ezPDF Builder Supreme', 'creator': 'PyPDF', 'creationdate': '2024-12-22T23:44:00+09:00', 'moddate': '2025-01-09T17:28:20+09:00', 'source': '/content/drive/MyDrive/data/rag_files/year_end_tax.pdf', 'total_pages': 426, 'page': 0, 'page_label': '1', 'page_number': 1}
연말정산 신고안내 일 하나는 제대로 하는, 국민께 인정받는 국세청 2024. 12. 2024 원천징수의무자를 위한 맞춤형 안내 간소화 서비스 일괄제공 서비스 발간등록번호 11-1210000-000072-10


## 4. 청킹


In [7]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=500,
    chunk_overlap=100,
    length_function=len,
)

all_splits = text_splitter.split_documents(cleaned_pages)

print(f"총 {len(all_splits)}개의 조각이 생성되었습니다.")
print(all_splits[0].metadata)
print(all_splits[0].page_content[:500])


총 1086개의 조각이 생성되었습니다.
{'producer': 'ezPDF Builder Supreme', 'creator': 'PyPDF', 'creationdate': '2024-12-22T23:44:00+09:00', 'moddate': '2025-01-09T17:28:20+09:00', 'source': '/content/drive/MyDrive/data/rag_files/year_end_tax.pdf', 'total_pages': 426, 'page': 0, 'page_label': '1', 'page_number': 1}
연말정산 신고안내 일 하나는 제대로 하는, 국민께 인정받는 국세청 2024. 12. 2024 원천징수의무자를 위한 맞춤형 안내 간소화 서비스 일괄제공 서비스 발간등록번호 11-1210000-000072-10


## 5. Vector DB & Retriever 생성


In [8]:
import gc
import os

os.environ.setdefault("PYTORCH_CUDA_ALLOC_CONF", "expandable_segments:True")

import faiss
import torch
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_community.docstore.in_memory import InMemoryDocstore
from langchain_community.vectorstores import FAISS

embedding_model_name = "nlpai-lab/KoE5"
use_gpu_for_document_embedding = True
embedding_device = "cuda" if use_gpu_for_document_embedding and torch.cuda.is_available() else "cpu"

# 문서 벡터 생성은 GPU를 잠깐 사용하고, FAISS 인덱스 자체는 CPU에 저장합니다.
doc_embeddings = HuggingFaceEmbeddings(
    model_name=embedding_model_name,
    model_kwargs={"device": embedding_device},
    encode_kwargs={"batch_size": 16},
)

print(f"문서 임베딩 device: {embedding_device}")

embedding_dim = len(doc_embeddings.embed_query(all_splits[0].page_content[:100]))
cpu_index = faiss.IndexFlatL2(embedding_dim)

vector_store = FAISS(
    embedding_function=doc_embeddings,
    index=cpu_index,
    docstore=InMemoryDocstore(),
    index_to_docstore_id={},
)

ids = vector_store.add_documents(documents=all_splits)
print(f"총 {len(ids)}개의 조각이 CPU 벡터 스토어에 추가되었습니다.")

# 이후 질문 임베딩은 CPU에서 수행해 LLM용 GPU 메모리를 남깁니다.
query_embeddings = HuggingFaceEmbeddings(
    model_name=embedding_model_name,
    model_kwargs={"device": "cpu"},
)

vector_store.embedding_function = query_embeddings
embeddings = query_embeddings

del doc_embeddings
gc.collect()

if torch.cuda.is_available():
    torch.cuda.empty_cache()
    print(f"임베딩 정리 후 GPU 할당 메모리: {torch.cuda.memory_allocated() / 1024**3:.2f} GiB")

retriever = vector_store.as_retriever(search_kwargs={"k": 3})


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:103: UserWarning: 
Error while fetching `HF_TOKEN` secret value from your vault: 'Requesting secret HF_TOKEN timed out. Secrets can only be fetched when running from the Colab UI.'.
You are not authenticated with the Hugging Face Hub in this notebook.
If the error persists, please let us know by opening an issue on GitHub (https://github.com/huggingface/huggingface_hub/issues/new).
  warnings.warn(


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/165 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

config.json:   0%|          | 0.00/741 [00:00<?, ?B/s]

...

special_tokens_map.json:   0%|          | 0.00/964 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/297 [00:00<?, ?B/s]

문서 임베딩 device: cuda
총 1086개의 조각이 CPU 벡터 스토어에 추가되었습니다.


Loading weights:   0%|          | 0/391 [00:00<?, ?it/s]

임베딩 정리 후 GPU 할당 메모리: 0.01 GiB


## 6. Retriever 작동 확인


In [9]:
import time

question = "월세 세액공제 조건은?"
start = time.time()
docs = retriever.invoke(question)
print(f"검색 시간: {time.time() - start:.2f}초")
print(f"검색된 문서 수: {len(docs)}")

for doc in docs:
    print(doc.metadata)
    print(doc.page_content[:300])
    print("-" * 80)


검색 시간: 0.22초
검색된 문서 수: 3
{'producer': 'ezPDF Builder Supreme', 'creator': 'PyPDF', 'creationdate': '2024-12-22T23:44:00+09:00', 'moddate': '2025-01-09T17:28:20+09:00', 'source': '/content/drive/MyDrive/data/rag_files/year_end_tax.pdf', 'total_pages': 426, 'page': 215, 'page_label': '216', 'page_number': 216}
이하자(종합소득금액 7천만원 이하자)：15% ○ 총급여액 5,500만원 이하자(종합소득금액 4,500만원 이하자)：17% 3) 월세액 세액공제 요건 국민주택규모의 주택 또는 기준시가가 4억원 이하인 주택[임대차계약증서의 주소지와 주민 등록표 등본의 주소지(외국인의 경우에는 ｢출입국관리법｣ 제32조제4호에 따른 국내 체류지 또는 ｢재외동포의 출입국과 법적 지위에 관한 법률｣ 제6조에 따라 신고한 국내거소)가 같아야 함]을 임차하기 위하여 지급하는 월세액(사글세액 포함) ※ 근로자 본인이 세액공제를 받기 위해서는 임대차계약증서의 주소지
--------------------------------------------------------------------------------
{'producer': 'ezPDF Builder Supreme', 'creator': 'PyPDF', 'creationdate': '2024-12-22T23:44:00+09:00', 'moddate': '2025-01-09T17:28:20+09:00', 'source': '/content/drive/MyDrive/data/rag_files/year_end_tax.pdf', 'total_pages': 426, 'page': 216, 'page_label': '217', 'page_number': 217}
Ⅵ. 세액감면(공제) 및 농어촌특별세 20

## 7. 모델 및 토크나이저 세팅


In [11]:
import gc
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig
from langchain_huggingface import ChatHuggingFace, HuggingFacePipeline

gc.collect()
if torch.cuda.is_available():
    torch.cuda.empty_cache()

# language_model_name = "nlpai-lab/KULLM3"
# 속도나 OOM 문제가 있으면 아래 5B 모델로 바꿔서 실행하세요.
language_model_name = "Bllossom/llama-3.2-Korean-Bllossom-AICA-5B"

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_use_double_quant=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.float16,
)

if torch.cuda.is_available():
    total_gib = torch.cuda.get_device_properties(0).total_memory // 1024**3
    gpu_limit_gib = max(int(total_gib) - 2, 8)
    max_memory = {0: f"{gpu_limit_gib}GiB", "cpu": "48GiB"}
else:
    max_memory = {"cpu": "48GiB"}

print(f"사용 모델: {language_model_name}")
print(f"max_memory 설정: {max_memory}")

model = AutoModelForCausalLM.from_pretrained(
    language_model_name,
    quantization_config=bnb_config,
    trust_remote_code=True,
    device_map="auto",
    max_memory=max_memory,
    offload_folder="./offload",
    offload_state_dict=True,
    low_cpu_mem_usage=True,
)
model.eval()

if hasattr(model, "hf_device_map"):
    print(model.hf_device_map)
else:
    print(f"모델 device: {next(model.parameters()).device}")


사용 모델: Bllossom/llama-3.2-Korean-Bllossom-AICA-5B
max_memory 설정: {0: '20GiB', 'cpu': '48GiB'}


Fetching 3 files:   0%|          | 0/3 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/346 [00:00<?, ?it/s]

MllamaForCausalLM LOAD REPORT from: Bllossom/llama-3.2-Korean-Bllossom-AICA-5B
Key                                                                                             | Status     |  | 
------------------------------------------------------------------------------------------------+------------+--+-
vision_model.global_transformer.layers.{0, 1, 2, 3, 4, 5, 6, 7}.mlp.fc2.weight                  | UNEXPECTED |  | 
vision_model.transformer.layers.{0...31}.self_attn.q_proj.weight                                | UNEXPECTED |  | 
vision_model.transformer.layers.{0...31}.mlp.fc1.weight                                         | UNEXPECTED |  | 
vision_model.transformer.layers.{0...31}.post_attention_layernorm.weight                        | UNEXPECTED |  | 
vision_model.transformer.layers.{0...31}.post_attention_layernorm.bias                          | UNEXPECTED |  | 
vision_model.transformer.layers.{0...31}.mlp.fc1.bias                                           | UNEXPECTED |  | 
v

모델 device: cuda:0


In [ ]:
from transformers import pipeline

llm_pipeline = pipeline(
    model=model,
    tokenizer=tokenizer,
    task="text-generation",
    do_sample=False,
    temperature=None,
    repetition_penalty=1.2,
    return_full_text=False,
    max_new_tokens=1000,
)

llm = HuggingFacePipeline(pipeline=llm_pipeline)
chat_model = ChatHuggingFace(llm=llm)


## 8. RAG 체인 구성


In [ ]:
from langchain_core.prompts import PromptTemplate
from langchain.schema.runnable import RunnablePassthrough
from langchain.schema.output_parser import StrOutputParser


template = """
아래에 주어진 맥락을 이용해 질문에 대해 답변해 줘.
주어진 맥락으로 답변이 어려운 상황이라면, 그냥 모른다고 답하고 억지로 답변을 꾸며 내지 마.
최대한 자세하게 답변해 줘.
반드시 한국어로 답변해야 해.
1000토큰 이내로 완성시켜줘.

맥락:
{context}

질문:
{question}
"""

prompt = PromptTemplate.from_template(template)


def format_docs(docs):
    formatted_docs = []

    for doc in docs:
        page_number = doc.metadata.get("page_number", "unknown")
        formatted_docs.append(
            f"[page {page_number}]\n{doc.page_content}"
        )

    return "\n\n".join(formatted_docs)


retrieval_chain = (
    {
        "context": retriever | format_docs,
        "question": RunnablePassthrough(),
    }
    | prompt
    | chat_model
    | StrOutputParser()
)


## 9. 기본 RAG 테스트


In [26]:
test_questions = [
    "연말 정산 때 비거주자가 주의할 점을 알려 줘.",
    "2024년 개정 세법 중에 월세와 관련한 내용이 있을까?",
    "기부금 공제 때 주의할 점은?",
]

for question in test_questions:
    print(f"질문: {question}")
    start = time.time()
    result = retrieval_chain.invoke(question)
    print(f"소요 시간: {time.time() - start:.2f}초")
    print(result)
    print("=" * 100)


Both `max_new_tokens` (=512) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


질문: 연말 정산 때 비거주자가 주의할 점을 알려 줘.


Both `max_new_tokens` (=512) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


소요 시간: 37.13초
물론입니다. 연말 정산 시 비거주자가 주의할 점들을 설명드리겠습니다.

### 1. 국내원천소득
- **근로소득**: 비거주자가 해외에서 근로받는 근로소득에 대해서는 소득세법 제122조에 따르면, 이 소득은 소득세에 과세됩니다. 그러나 특정 조건이 충족될 경우, 이를 공제할 수도 있습니다.
    - **특수조치**: 소득세법 제51조 제3항에서는 "국내 사업장 없이"라는 단어가 포함되어 있어, 비거주자가 해외에서 근로받는 소득이 소득세에 과세되지는 않다는 규정이 있지만, 이는 국외 근로소득에 한해서 적용됩니다.
    - **중복공제**: 동일한 근로소득이 여러 국가에서 발생했을 경우, 각국의 정부로부터 받은 공제나 혜택을 바탕으로 중복공제를 신청할 수 있습니다.

### 2. 법인세 관련 사항
- **회사에서의 업무 수행**: 비거주자가 회사가 해외에서 운영되고 있다면, 회사의 활동 결과에 영향을 미치는 업무를 수행했다는 이유로, 회사의 법인세 신고서에 반영하도록 해야 합니다.
    - **증명서류 확인**: 회사가 제출한 증명서류를 철저히 검토하여 모든 내용이 정확하고 적절한지 확인합니다. 특히, 회사의 벤처 관리 계획(BMP)에 명시된 정보가 중요한데요.
    - **특별한 혜택**: 법인세법 제52조에 의해 지정된 특별한 혜택이나 공제가 가능하다면, 이를 활용하여 세액을 줄일 수 있습니다.

### 3. 자산 거래
- **국내 자산 거래**: 비거주자가 국내에서 자산을 매매하면, 해당 매출/매입에 대해 소득세를 과세할 수 있습니다.
    - **증명서류**: 자산 거래와 관련된 증명서류를 잘 작성하고 제출해야 합니다. 여기에는 매매 계약서, 거래 당사자 목록, 거래 날짜 등을 포함합니다.
    - **공제 가능한 부분**: 자산 판매로 인한 손실을 공제할 수 있는 경우가 있을 수 있으며, 이를 위해 필요한 서류를 준비하세요
질문: 2024년 개정 세법 중에 월세와 관련한 내용이 있을까?


Both `max_new_tokens` (=512) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


소요 시간: 36.98초
네, 2024년에 개정된 세법에서는 월세와 관련한 몇 가지 주요 변경 사항이 있습니다. 특히, 조세특례제한법 제95조의2와 제122조의3에 따라 다음과 같은 변화가 있었습니다:

1. **월세 공제**: 2024년에는 월세 공제 규정이 강화되었습니다. 이는 특정 조건을 충족하는 사람들에게만 적용될 수 있으며, 주택 구매나 임차 시 발생되는 월세는 최대 한도를 넘어서는 부분까지 공제할 수 없습니다.

2. **소득 기준 및 한도 상향**: 2024년에는 소득 기준과 한도가 상승했습니다. 이를 통해 세금 부담이 약화됩니다. 예를 들어, 총급여 7천만원 이하의 경우, 무주택 근로자는 그 범위 내에서 월세 공제를 할 수 있고, 총급여 8천만원 이하의 경우에도 동일한 방식이 적용됩니다.

3. **국민 주택 규모 및 기준 시장 가격**: 2024년에는 국민 주택 규모와 기준 시장 가격이 다시 재평가되어, 이러한 기준에 맞추게 월세 공제가 가능합니다. 즉, 국민 주택 규모 85㎡ 이하 또는 기준 시가 4억원 이하인 주택에 대한 월세 공제가 가능합니다.

4. **신용 카드 소득 공제율 상향**: 또한, 신용카드 소득 공제율도 상향되었으며, 이는 전체 소득 공제에 영향을 미칠 것입니다. 

따라서, 2024년에는 월세와 관련된 여러 측면에서 법률적 변화를 보았습니다. 이를 잘 이해하면 개인에게 유익한 세금 혜택을 누릴 수 있습니다.assistant

네, 2024년에 개정된 세법에서는 월세와 관련한 몇 가지 주요 변경 사항이 있습니다. 특히, 조세특례제한법 제95조의2와 제122조의3에 따라 다음과 같은 변화가 있었습니다:

1. **월세 공제**: 2024년에는 월세 공제 규정이 강화되었습니다. 이는 특정 조건을 충족하는 사람들에게만 적용될 수 있으며, 주택 구매나 임차 시 발생되는 월세는 최대 한도를 넘어서는 부분까지 공제할 수 없습니다.

2
질문: 기부금 공제 때 주의할 점은?
소요 시간: 37.01초
기부금 공제 할 때 주의할 점은 다음과

## 10. 심화 RAG 실험 준비


In [ ]:
# 심화 RAG에서 사용할 추가 패키지입니다.
# rank_bm25: Hybrid Search(BM25)용
# sentence-transformers: CrossEncoder Reranker용
!pip install -q rank_bm25 sentence-transformers


In [ ]:
import os
import re
import time
import pandas as pd
from langchain_core.runnables import RunnableLambda

preview_question = "2024년 개정 세법 중에 월세와 관련한 내용이 있을까?"


def get_source_pages(docs):
    return sorted({
        doc.metadata.get("page_number")
        for doc in docs
        if doc.metadata.get("page_number") is not None
    })


def answer_from_docs(question, docs, target_model=None):
    if target_model is None:
        target_model = chat_model

    return (
        prompt
        | target_model
        | StrOutputParser()
    ).invoke({
        "context": format_docs(docs),
        "question": question,
    })


def run_rag_experiment(method_name, target_retriever, questions=None, target_model=None):
    if questions is None:
        questions = test_questions

    rows = []

    for question in questions:
        print(f"[{method_name}] 질문: {question}")
        start = time.time()
        docs = []
        pages = []

        try:
            docs = target_retriever.invoke(question)
            answer = answer_from_docs(question, docs, target_model=target_model)
            pages = get_source_pages(docs)
        except Exception as e:
            answer = f"ERROR: {type(e).__name__}: {e}"

        elapsed = round(time.time() - start, 2)

        rows.append({
            "method": method_name,
            "question": question,
            "elapsed_sec": elapsed,
            "source_pages": pages,
            "answer": answer,
        })

        print(f"소요 시간: {elapsed}초")
        print(f"참고 페이지: {pages}")
        print(answer)
        print("=" * 100)

    return pd.DataFrame(rows)


def show_retrieval_results(method_name, target_retriever, question=preview_question):
    docs = target_retriever.invoke(question)
    print(f"[{method_name}] 검색 결과 확인")
    print(f"질문: {question}")
    print(f"검색 문서 수: {len(docs)}")
    print(f"참고 페이지: {get_source_pages(docs)}")
    print("-" * 100)

    for i, doc in enumerate(docs, start=1):
        print(f"문서 {i} / page {doc.metadata.get('page_number', 'unknown')}")
        print(doc.page_content[:400])
        print("-" * 100)

    return docs


## 11. Hybrid Search: BM25 + Vector Search


In [ ]:
from langchain_community.retrievers import BM25Retriever
from langchain.retrievers import EnsembleRetriever

# 키워드 검색(BM25)과 의미 검색(FAISS)을 섞습니다.
# 세법 문서처럼 정확한 용어가 중요한 문서에서는 BM25가 보완 역할을 합니다.
bm25_retriever = BM25Retriever.from_documents(all_splits)
bm25_retriever.k = 5

faiss_retriever = vector_store.as_retriever(search_kwargs={"k": 5})

hybrid_retriever = EnsembleRetriever(
    retrievers=[bm25_retriever, faiss_retriever],
    weights=[0.4, 0.6],
)

hybrid_preview_docs = show_retrieval_results("Hybrid Search", hybrid_retriever)


In [ ]:
hybrid_preview_df = run_rag_experiment(
    "Hybrid Search",
    hybrid_retriever,
    questions=[preview_question],
)
hybrid_preview_df


## 12. Multi-Query Retrieval


In [ ]:
multi_query_prompt = PromptTemplate.from_template("""
아래 질문을 RAG 검색에 유리하도록 서로 다른 표현의 한국어 검색 질문 3개로 바꿔줘.
번호나 설명 없이 질문만 한 줄에 하나씩 작성해.

원래 질문:
{question}
""")


def parse_multi_queries(raw_text, original_question, max_queries=3):
    queries = []

    for line in raw_text.splitlines():
        line = line.strip()
        line = re.sub(r"^[\-\*•\d\.\)\s]+", "", line).strip()

        if not line:
            continue
        if line in queries or line == original_question:
            continue
        if len(line) > 120:
            line = line[:120].strip()

        queries.append(line)

        if len(queries) >= max_queries:
            break

    return [original_question] + queries


def make_multi_queries(question):
    raw_queries = (
        multi_query_prompt
        | chat_model
        | StrOutputParser()
    ).invoke({"question": question})

    queries = parse_multi_queries(raw_queries, question)

    print("생성된 검색 질문:")
    for query in queries:
        print("-", query)

    return queries


def multi_query_search(question, per_query_k=3, final_k=5):
    queries = make_multi_queries(question)
    seen = set()
    merged_docs = []

    for query in queries:
        docs = faiss_retriever.invoke(query)

        for doc in docs[:per_query_k]:
            key = (
                doc.metadata.get("source"),
                doc.metadata.get("page_number"),
                doc.page_content[:120],
            )

            if key in seen:
                continue

            seen.add(key)
            merged_docs.append(doc)

    return merged_docs[:final_k]


multi_query_retriever = RunnableLambda(lambda question: multi_query_search(question))
multi_query_preview_docs = show_retrieval_results("Multi-Query", multi_query_retriever)


In [ ]:
multi_query_preview_df = run_rag_experiment(
    "Multi-Query",
    multi_query_retriever,
    questions=[preview_question],
)
multi_query_preview_df


## 13. Reranking


In [ ]:
from sentence_transformers import CrossEncoder

# Reranker는 후보 문서를 다시 점수화합니다.
# LLM과 GPU 메모리 충돌을 피하려고 기본값은 CPU입니다. 큰 GPU를 쓰면 device="cuda"로 바꿔도 됩니다.
reranker = CrossEncoder(
    "BAAI/bge-reranker-v2-m3",
    device="cpu",
)

rerank_candidate_retriever = hybrid_retriever


def rerank_search(question, top_n=3):
    candidate_docs = rerank_candidate_retriever.invoke(question)

    if not candidate_docs:
        return []

    pairs = [(question, doc.page_content) for doc in candidate_docs]
    scores = reranker.predict(pairs)

    ranked_docs = sorted(
        zip(candidate_docs, scores),
        key=lambda item: item[1],
        reverse=True,
    )

    print("Rerank 점수:")
    for doc, score in ranked_docs[:top_n]:
        print(f"page {doc.metadata.get('page_number', 'unknown')} / score {score:.4f}")

    return [doc for doc, score in ranked_docs[:top_n]]


rerank_retriever = RunnableLambda(lambda question: rerank_search(question))
rerank_preview_docs = show_retrieval_results("Reranking", rerank_retriever)


In [ ]:
rerank_preview_df = run_rag_experiment(
    "Reranking",
    rerank_retriever,
    questions=[preview_question],
)
rerank_preview_df


## 14. Contextual Compression


In [ ]:
from langchain.retrievers import ContextualCompressionRetriever
from langchain.retrievers.document_compressors import EmbeddingsFilter

# 검색된 문서 중 질문과 임베딩 유사도가 높은 문서만 남깁니다.
# 여기서는 문장 단위 압축보다는 문서 필터링 방식으로 가볍게 적용합니다.
compression_filter = EmbeddingsFilter(
    embeddings=embeddings,
    k=3,
)

compression_retriever = ContextualCompressionRetriever(
    base_compressor=compression_filter,
    base_retriever=hybrid_retriever,
)

compression_preview_docs = show_retrieval_results("Contextual Compression", compression_retriever)


In [ ]:
compression_preview_df = run_rag_experiment(
    "Contextual Compression",
    compression_retriever,
    questions=[preview_question],
)
compression_preview_df


## 15. LLM 비교: OpenAI API 선택 실험


In [ ]:
from langchain_openai import ChatOpenAI

# OPENAI_API_KEY가 환경변수에 있을 때만 실행합니다.
# 모델명은 계정 접근 권한에 따라 바꿀 수 있습니다.
OPENAI_MODEL = os.getenv("OPENAI_MODEL", "gpt-5-mini")
openai_chat_model = None

if os.getenv("OPENAI_API_KEY"):
    try:
        openai_chat_model = ChatOpenAI(
            model=OPENAI_MODEL,
            temperature=0,
        )
        print(f"OpenAI 비교 모델 준비 완료: {OPENAI_MODEL}")
    except Exception as e:
        print(f"OpenAI 모델 준비 실패: {e}")
else:
    print("OPENAI_API_KEY가 없어 OpenAI 비교 실험은 건너뜁니다.")
    print("실험하려면 os.environ['OPENAI_API_KEY']를 설정한 뒤 이 셀을 다시 실행하세요.")


In [ ]:
if openai_chat_model is not None:
    openai_hybrid_preview_df = run_rag_experiment(
        f"OpenAI {OPENAI_MODEL} + Hybrid",
        hybrid_retriever,
        questions=[preview_question],
        target_model=openai_chat_model,
    )
    display(openai_hybrid_preview_df)
else:
    print("OpenAI 비교 모델이 없어 이 셀은 실행하지 않습니다.")


## 16. 최종 비교


In [ ]:
experiment_configs = [
    ("기본 RAG", retriever, chat_model),
    ("Hybrid Search", hybrid_retriever, chat_model),
    ("Multi-Query", multi_query_retriever, chat_model),
    ("Reranking", rerank_retriever, chat_model),
    ("Contextual Compression", compression_retriever, chat_model),
]

if openai_chat_model is not None:
    experiment_configs.append((f"OpenAI {OPENAI_MODEL} + Hybrid", hybrid_retriever, openai_chat_model))

comparison_rows = []

for method_name, target_retriever, target_model in experiment_configs:
    result_df = run_rag_experiment(
        method_name,
        target_retriever,
        questions=test_questions,
        target_model=target_model,
    )
    comparison_rows.append(result_df)

comparison_df = pd.concat(comparison_rows, ignore_index=True)
comparison_df["answer_preview"] = comparison_df["answer"].str.slice(0, 250)
comparison_df[["method", "question", "elapsed_sec", "source_pages", "answer_preview"]]


In [ ]:
# 필요하면 전체 답변을 방법별로 펼쳐서 확인합니다.
for _, row in comparison_df.iterrows():
    print(f"[{row['method']}] {row['question']}")
    print(f"소요 시간: {row['elapsed_sec']}초 / 참고 페이지: {row['source_pages']}")
    print(row["answer"])
    print("=" * 120)
